# Stuff- : محرك الأسلوب الجديد (Remotion + three.js)
شغّل الخلايا بالترتيب. **الرندر يشغّله المستخدم فقط**، ولا يُشغَّل في أي بيئة أخرى.
- اختر `Runtime > Change runtime type > T4 GPU` (بدون GPU الرندر يستغرق ساعات).
- الخلية 2 تفحص هل يصل WebGL إلى كرت الشاشة، وإن لم يصل تحاول تفعيله تلقائيًا (تنزيل نحو 400 ميغابايت). إن بقي `None` فأرسل لي ناتجها كاملًا.
- بعد أي تحديث أعدّ تشغيل الخلية 1 أولًا (تسحب آخر نسخة).
- المدخل الوحيد: السكريبت الإنجليزي في الخلية 3.

In [ ]:
# 1) Setup (safe to re-run; installs once)
import os
os.chdir("/content")
REPO = "/content/Stuff-"
def node_major():
    import subprocess
    try: return int(subprocess.check_output(["node", "-v"]).decode().strip()[1:].split(".")[0])
    except Exception: return 0
if not os.path.exists("/content/.engine_installed"):
    !rm -rf {REPO}
    !git clone -q https://github.com/marymyc/Stuff-.git {REPO}
    !apt-get -qq install -y ffmpeg fonts-dejavu-core libsndfile1 > /dev/null
    if node_major() < 18:
        !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
        !apt-get install -y -qq nodejs > /dev/null
    !cd {REPO}/engine && npm install --no-audit --no-fund --loglevel=error 2>&1 | tail -3
    print("Installing Python packages (Chatterbox + torch 2.6, several minutes)...")
    !pip install --progress-bar off -q -e "{REPO}[tts]" 2>&1 | tail -n 3
    open("/content/.engine_installed", "w").write("ok")
else:
    !git -C {REPO} pull -q
!pip uninstall -y -q torchvision
os.chdir(REPO)
print("ready:", os.getcwd(), "| node", node_major())

In [ ]:
# 2) Chromium + can WebGL reach the REAL GPU?  If not, try to enable it (downloads the matching NVIDIA driver, ~400 MB)
import os, re, subprocess
os.chdir("/content")
!pip install -q playwright
!playwright install --with-deps chromium 2>&1 | tail -2
def probe():
    out = subprocess.run(["python", "/content/Stuff-/engine/tools/gpu_probe.py"], capture_output=True, text=True)
    print(out.stdout, out.stderr[-500:])
    m = re.search(r"RESULT_GL=(\w+)", out.stdout)
    return m.group(1) if m and m.group(1) != "NONE" else None
GL = probe()
if GL is None:
    print("\n>>> No real GPU path yet. Trying to enable NVIDIA graphics libraries ...")
    print(subprocess.run(["python", "/content/Stuff-/engine/tools/enable_gpu_gl.py"], capture_output=True, text=True).stdout)
    GL = probe()
print("GL backend to use:", GL)
os.chdir("/content/Stuff-")

In [ ]:
# 3) YOUR INPUT
SCRIPT = """Paste your English script here (about 150-170 words for 60-70 seconds)."""
NAME = "short1"
REFERENCE_WAV = ""      # optional: clean 10+ s American-male clip you have rights to
SCALE = 1               # 1 = 1080x1920, 1.3333 = 2K
FRAMES = ""             # e.g. "0-89" to render only the first 3 seconds as a preview

import os
os.chdir("/content/Stuff-")
open("/content/script.txt", "w").write(SCRIPT)
print(len(SCRIPT.split()), "words")

In [ ]:
# 4) Make the video (voice -> plan -> render -> mux). Progress prints below.
import glob, os, subprocess
os.chdir("/content/Stuff-/engine")
assert GL, "No real GPU path: stop here (see cell 2)."
chrome = sorted(glob.glob(os.path.expanduser("~/.cache/ms-playwright/chromium-*/chrome-linux*/chrome")))[-1]
env = dict(os.environ, CHROME_PATH=chrome, REMOTION_GL=GL, REMOTION_TMP="/content/rtmp")
cmd = ["python", "tools/make.py", "/content/script.txt", "--name", NAME, "--scale", str(SCALE), "--gl", GL]
if REFERENCE_WAV: cmd += ["--ref-voice", REFERENCE_WAV]
if FRAMES: cmd += ["--frames", FRAMES]
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
print("exit code:", p.wait())

In [ ]:
# 5) Download
from google.colab import files
files.download(f"/content/Stuff-/engine/out/{NAME}/final.mp4")